# GeoResistPy — Quick-Start Demo

This notebook walks through the core API:
1. Import data
2. Quality control
3. Generate electrode array
4. 1-D forward modelling
5. 1-D inversion
6. Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import georesistpy modules
from georesistpy.io.readers import read_csv
from georesistpy.qc.filters import remove_negative_resistivity, filter_outliers_mad
from georesistpy.qc.errors import estimate_error_model
from georesistpy.utils.arrays import generate_array, electrode_positions
from georesistpy.forward.forward1d import forward_1d
from georesistpy.inversion.inversion1d import invert_1d
from georesistpy.visualization.plots import plot_pseudosection

## 1. Import Data

In [ ]:
df = read_csv('../examples/sample_ves.csv')
print(f'Loaded {len(df)} measurements')
df.head()

## 2. Quality Control

In [ ]:
df = remove_negative_resistivity(df)
df = filter_outliers_mad(df, threshold=3.5)
df = estimate_error_model(df)
print(f'After QC: {len(df)} measurements')
df.head()

## 3. Generate Electrode Array

In [ ]:
scheme = generate_array('wenner', n_electrodes=24, spacing=1.0)
pos = electrode_positions(24, spacing=1.0)
print(f'Generated {len(scheme)} configurations')

plt.figure(figsize=(10, 2))
plt.scatter(pos[:, 0], pos[:, 1], marker='v', s=60, c='navy')
plt.xlabel('Distance (m)')
plt.title('Electrode Positions')
plt.show()

## 4. 1-D Forward Model

In [ ]:
# Define a 4-layer model
rho_true = [100, 50, 200, 500]
thk_true = [2, 5, 10]
ab2 = np.array([1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 70, 100])

rhoa_syn = forward_1d(rho_true, thk_true, ab2)

plt.figure(figsize=(8, 5))
plt.loglog(ab2, rhoa_syn, 'bo-')
plt.xlabel('AB/2 (m)')
plt.ylabel('Apparent Resistivity (Ω·m)')
plt.title('Synthetic VES Sounding')
plt.grid(True, which='both', ls='--', alpha=0.4)
plt.show()

## 5. 1-D Inversion

In [ ]:
# Add 3% noise
np.random.seed(42)
rhoa_noisy = rhoa_syn * (1 + 0.03 * np.random.randn(len(rhoa_syn)))

result = invert_1d(
    ab2=ab2,
    rhoa_obs=rhoa_noisy,
    n_layers=4,
    lam=20,
    max_iter=30,
)

print(f'RMS: {result.rms:.2f}%')
print(f'Resistivities: {result.resistivity}')
print(f'Thicknesses:   {result.thickness}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Sounding
axes[0].loglog(ab2, rhoa_noisy, 'ko', label='Observed')
axes[0].loglog(ab2, result.response, 'r-', lw=2, label='Fitted')
axes[0].set_xlabel('AB/2 (m)')
axes[0].set_ylabel('ρₐ (Ω·m)')
axes[0].legend()
axes[0].grid(True, which='both', ls='--', alpha=0.4)
axes[0].set_title('Sounding Curve Fit')

# Model
depths = np.concatenate([[0], np.cumsum(result.thickness)])
for i, r in enumerate(result.resistivity):
    d_top = depths[i]
    d_bot = depths[i+1] if i+1 < len(depths) else d_top + result.thickness[-1]
    axes[1].barh((d_top+d_bot)/2, r, height=d_bot-d_top,
                 align='center', alpha=0.7, edgecolor='k')
axes[1].set_xlabel('Resistivity (Ω·m)')
axes[1].set_ylabel('Depth (m)')
axes[1].invert_yaxis()
axes[1].set_xscale('log')
axes[1].set_title('Inverted Model')
plt.tight_layout()
plt.show()

## 6. Pseudosection (2-D data)

In [ ]:
df_2d = read_csv('../examples/sample_ert2d.csv')
fig = plot_pseudosection(df_2d)
plt.show()